In [1]:
import sys, os
# sys.path.append(r"c:\Users\Preet Lodaya\Valiance_EDA")
from langchain.agents import initialize_agent, AgentType
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from tools.data_analysis_tools import (
      DescribeData,
      CorrelationAnalysis,
      VisualizeData,
      CalculateWMAPE
)
import ast
from langchain_google_vertexai import GemmaLocalHF, GemmaChatLocalHF

from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain.tools import Tool as LangTool
from langchain.prompts import ChatPromptTemplate


c:\Users\Preet Lodaya\Valiance_EDA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pwd!

'c:\\Users\\Preet Lodaya\\Valiance_EDA\\ui'

In [4]:
tools = [
        DescribeData(),
                CorrelationAnalysis(),
                VisualizeData(),
                CalculateWMAPE()
            ]
def find_tool(name: str):
    return next((t for t in tools if getattr(t, "name", t.__class__.__name__) == name), None)

# # Prepare dataset summary for prompt
# numeric_cols = self.df.select_dtypes(include=['number']).columns.tolist()
# categorical_cols = self.df.select_dtypes(include=['object', 'category']).columns.tolist()
# dataset_summary = f"DATASET: {self.dataset_name} ({self.df.shape[0]} rows × {self.df.shape[1]} columns)\nNumeric cols: {numeric_cols}\nCategorical cols: {categorical_cols}\nSample:\n{self.df.head(3).to_string()}\n"

langchain_tools = []
for t in tools:
    tname = getattr(t, "name", t.__class__.__name__)
    tdesc = getattr(t, "description", "")
    # wrap tool call into a simple function that accepts kwargs
    def make_fn(tool_obj):
        def _fn(text: str = ""):
            # Many of our tools expect structured args; this wrapper keeps backward compatibility.
            # If text is a JSON/dict-like string, try to parse to dict via literal_eval.
            try:
                parsed = ast.literal_eval(text) if text else {}
            except Exception:
                parsed = {"text": text}
            try:
                if isinstance(parsed, dict):
                    return tool_obj._run(**parsed)
                if isinstance(parsed, (list, tuple)):
                    return tool_obj._run(*parsed)
                return tool_obj._run(parsed)
            except TypeError:
                # fallback: try passing raw text
                return tool_obj._run(text)
        return _fn
    langchain_tools.append(LangTool(name=tname, func=make_fn(t), description=tdesc))

print(langchain_tools)

[Tool(name='describe_data', description='Get statistical description of the dataframe or a specific column.', func=<function make_fn.<locals>._fn at 0x0000025F561F0F40>), Tool(name='correlation_analysis', description='Compute correlation matrix for numeric columns and return a heatmap image as a markdown data URL.', func=<function make_fn.<locals>._fn at 0x0000025F561F0D60>), Tool(name='visualize_data', description='Create visualizations: histogram, scatter, box, bar. Returns a PNG image as a markdown data URL.', func=<function make_fn.<locals>._fn at 0x0000025F561F0CC0>), Tool(name='calculate_WMAPE', description="Calculate WMAPE grouped by 'intersection'. Signature: forecast_col, actuals_col.", func=<function make_fn.<locals>._fn at 0x0000025F561F0E00>)]


In [ ]:

# Build the decision prompt
system_prompt = f"""
You are a data analysis agent. For the user query below you must choose ONE of the following options (and return only that):

1) Call a tool using EXACT markup (no extra text):
   <tool>tool_name</tool><args>{{"arg1": "value1", ...}}</args>

   Available tools:
   - describe_data: Get statistical description of the data
   - correlation_analysis: Correlation matrix and heatmap
   - visualize_data: Create plots (histogram, scatter, box, bar)
   - calculate_WMAPE: Calculate WMAPE grouped by intersection

OR

2) Return a Python code block (```python ... ```) that uses the dataframe variable `df` and common libraries (pandas, numpy, matplotlib, seaborn) to perform analysis and produce textual or visual output.

Do NOT mix tool markup and code. The dataset summary is below.


USER QUESTION: "calculate the wmape for each intersection"

Respond with EITHER a single tool call markup OR a single python code block.
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),  
])

system_prompt = """
You are a data analysis agent. 
For the user query below you must choose ONE of the following options:

1) Call a tool using EXACT markup (no extra text):
   <tool>tool_name</tool><args>{{"arg1": "value1", ...}}</args>

   Available tools: {tool_names}

   Tool descriptions:
   {tools}

OR

2) Return a Python code block (```python ... ```).

Do NOT mix tool markup and code.

USER QUESTION: "{input}"

{agent_scratchpad}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt)
])


llm = GemmaLocalHF(
    model_name="google/gemma-3-270m",
    hf_access_token="your token",
)


You are using a model of type gemma3_text to instantiate a model of type gemma. This is not supported for all configurations of models and can yield errors.


In [19]:
agent = create_react_agent(llm, tools = langchain_tools, prompt = None)
agent_executor = AgentExecutor(agent = agent, tools=langchain_tools, verbose=True, handle_parsing_errors=True, max_iterations=3)
agent_executor.invoke({"input":"calculate the wmape for each intersection"})
# decision = self._ask_model(agent_executor, prompt)

AttributeError: 'NoneType' object has no attribute 'input_variables'

In [18]:

agent = create_react_agent(llm, tools = langchain_tools, prompt = prompt)
agent_executor = AgentExecutor(agent = agent, tools=langchain_tools, verbose=True, handle_parsing_errors=True, max_iterations=3)
agent_executor.invoke({"input":"calculate the wmape for each intersection"})
# decision = self._ask_model(agent_executor, prompt)
                



> Entering new AgentExecutor chain...
System: 
You are a data analysis agent. 
For the user query below you must choose ONE of the following options:

1) Call a tool using EXACT markup (no extra text):
   <tool>tool_name</tool><args>{"arg1": "value1", ...}</args>

   Available tools: describe_data, correlation_analysis, visualize_data, calculate_WMAPE

   Tool descriptions:
   describe_data(text: str = '') - Get statistical description of the dataframe or a specific column.
correlation_analysis(text: str = '') - Compute correlation matrix for numeric columns and return a heatmap image as a markdown data URL.
visualize_data(text: str = '') - Create visualizations: histogram, scatter, box, bar. Returns a PNG image as a markdown data URL.
calculate_WMAPE(text: str = '') - Calculate WMAPE grouped by 'intersection'. Signature: forecast_col, actuals_col.

OR

2) Return a Python code block (```python ... ```).

Do NOT mix tool markup and code.

USER QUESTION: "calculate the wmape for each i

{'input': 'calculate the wmape for each intersection',
 'output': 'Agent stopped due to iteration limit or time limit.'}

In [9]:
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_HUB_TOKEN")
print(hf_token)

None
